# None

`None` is not just "nothing". It is a deliberate sentinel used to represent absence, missing configuration, uninitialized state, or no return value. Confusing `None` with empty values causes production bugs that are hard to spot.


## 1. Problem First

Why does this matter in real systems?

- `None` often means missing config, not false.
- APIs distinguish omitted values from empty strings and zero.
- Accidental `None` returns propagate into late failures.


In [ ]:
config_timeout = None

if config_timeout is None:
    config_timeout = 30

print(config_timeout)


## 2. Minimal Concept

`None` is a singleton object.

- Check it with `is None`, not `== None`
- Functions without explicit `return` produce `None`
- It is often used as a sentinel for absence


## 3. Mental Model

How Python actually behaves

`None` is a real object with a single shared identity. It often means absence, which is semantically different from empty containers, zero, or `False`.


In [ ]:
first = None
second = None
print(id(first), id(second))
print(first is second)


## 4. Internal Mechanics

A function with no explicit return emits `None`. That makes forgotten returns runtime logic bugs, not syntax bugs.


In [ ]:
import dis

def log_event(message):
    print(message)

print(log_event("boot"))
dis.dis(log_event)


## 5. Edge Cases

Where it breaks

- `None`, `False`, `0`, and `""` are all falsey but mean different things.
- Implicit `None` returns cause failures later.
- Using `or` for defaults can erase valid falsey values.


In [ ]:
port = 0
print(port or 8080)

missing = None
print(missing or 8080)


## 6. Performance Thinking

- Checking identity with `is None` is O(1).
- The performance story is trivial; the correctness story is not.
- `None` is cheap in memory but expensive when its meaning is ambiguous.


## 7. Real Use Case

A CLI loader reads optional config from disk. Missing config should trigger defaults, while explicit values should pass through unchanged.


In [ ]:
def resolve_region(config_region):
    if config_region is None:
        return "us-east-1"
    return config_region

print(resolve_region(None))
print(resolve_region("eu-west-1"))


## 8. Anti-Pattern

What beginners do wrong

- Use `== None` instead of `is None`.
- Collapse `None` and empty values into one case.
- Forget explicit returns and debug the failure downstream.


In [ ]:
def build_payload(user_id):
    if user_id:
        return {"user_id": user_id}
    # Implicit None here is dangerous

print(build_payload(0))


## 9. Interview Signals

Questions FAANG asks

- Why should `None` usually be checked with `is`?
- What is the semantic difference between `None` and an empty list?
- How do implicit `None` returns create bugs?
- When is `None` a good sentinel?


## 10. Exercise (Non-trivial)

Design a config merge function for a CLI tool where incoming values may be `None`, zero, empty strings, or explicit booleans. Decide which mean "missing" and which mean "intentionally provided." 


In [ ]:
def merge_timeout(cli_value, file_value, default=30):
    return cli_value or file_value or default

# Task:
# 1. Preserve valid zero values if the system allows them.
# 2. Treat None as missing.
# 3. Explain the meaning of each branch.
